# Spotify Lyrics Cleaning



### Goal
Turn `data/raw/spotify_millsongdata.csv` into a **clean, deduplicated, normalized** lyrics table that’s easy to align to Spotify tracks later.

### Input
- `data/raw/spotify_millsongdata.csv`

### Outputs
- `data/cleaned/lyrics_cleaned.csv`

### 1. Setup & Data Loading

#### 1.1 Path Configuration

In [1]:
import os 

IS_KAGGLE = os.path.exists("/kaggle/input")

if IS_KAGGLE:
    DATA_PATH = "/kaggle/input/spotify-million-song-dataset/spotify_millsongdata.csv"
    OUTPUT_PATH = "/kaggle/working/"
else:
    DATA_PATH = "../data/raw/spotify_millsongdata.csv"
    OUTPUT_PATH = "../data/cleaned/"

#### 1.2 Import required packages

In [ ]:
import re
import unicodedata
from pathlib import Path
import numpy as np
import pandas as pd

In [3]:
df = pd.read_csv(DATA_PATH)
df.shape, df.columns.tolist()

((57650, 4), ['artist', 'song', 'link', 'text'])

In [4]:
df.head(3)

,artist,song,link,text
0,ABBA,Ahe's My Kind Of Girl,/a/abba/ahes+my+kind+of+girl_20598417.html,"Look at her face, it's a wonderful face \r\nA..."
1,ABBA,"Andante, Andante",/a/abba/andante+andante_20002708.html,"Take it easy with me, please \r\nTouch me gen..."
2,ABBA,As Good As New,/a/abba/as+good+as+new_20003033.html,I'll never know why I had to go \r\nWhy I had...


### 2. Data Cleaning

In [6]:
# Basic QA
print(df.isna().mean().sort_values(ascending=False).head(10))
print("dup rows:", df.duplicated().sum())
print("dup artist+song:", df.duplicated(["artist", "song"]).sum())

artist    0.0
song      0.0
link      0.0
text      0.0
dtype: float64
dup rows: 0
dup artist+song: 2


Since there is no missing values, no duplicated rows and only 1 duplicate artist + song, there is not much cleaning to be done.

### 3. Normalization

#### 3.1 Define Regular Expressions

1. `_RE_FEAT`: matches “feature credits” like feat. ..., ft ..., or featuring ... (case-insensitive), including everything after it to the end of the string.

    Example: `"Drake feat. Rihanna"` → it matches `" feat. Rihanna"` so you can remove it.

2. `_RE_PARENS`: matches any text inside parentheses (...) or brackets [...] (non-greedy), including surrounding whitespace.

    Example: `"Song Title (Remastered 2011)"` → it matches `"(Remastered 2011)"` so you can strip it.

3. `_RE_NON_ALNUM`: matches any character that is not a lowercase letter a-z, digit 0-9, or whitespace `\s`.

    Example: `"Beyoncé!"` (after lowercasing/diacritics handling) → it matches punctuation like `!` so you can replace it with spaces and normalize tokens.

In [7]:
_RE_FEAT = re.compile(r"\s*\b(?:feat\.?|ft\.?|featuring)\b.*$", flags=re.IGNORECASE)
_RE_PARENS = re.compile(r"\s*[\(\[].*?[\)\]]\s*")
_RE_NON_ALNUM = re.compile(r"[^a-z0-9\s]+")

#### 3.2 Remove accents/diacritics

It removes accents/diacritics from a string (e.g., turns “Beyoncé” into “Beyonce”), while keeping the base letters.  

- `unicodedata.normalize("NFKD", s)`: converts characters into a decomposed form where accented letters are split into:
    1. a base character + a combining accent mark
    
        Example: "é" → "e" + "◌́" (combining acute accent)

- if not `unicodedata.combining(ch)`: filters out those combining accent marks.

- `"".join(...)`: stitches the remaining base characters back into a normal string.

&nbsp;  
**EXAMPLE:**  
&nbsp;  
Input: `"Señorita"` → Output: `"Senorita"`  
&nbsp;  
Input: `"Björk"` → Output: `"Bjork"`

In [8]:
def _strip_accents(s: str) -> str:
    return "".join(
        ch for ch in unicodedata.normalize("NFKD", s) if not unicodedata.combining(ch)
    )

#### 3.3 Normalize Artists

It converts an artist name into a standardized “join key” format so the same artist is more likely to match across datasets (Spotify vs lyrics).  

- **Handle missing input**: if `s is None` or becomes empty after stripping → return `""`.

- **Lowercase + trim**: makes matching case-insensitive and removes leading/trailing spaces.

- **Remove feature credits** (`_RE_FEAT.sub("", s)`): strips parts like `feat. ..., ft ..., featuring ...` if present (common in Spotify-style strings).

- **Remove accents** (`_strip_accents`): e.g., Beyoncé → Beyonce.

- **Replace punctuation/symbols with spaces** (`_RE_NON_ALNUM.sub(" ", s)`): keeps only a-z, 0-9, and whitespace.

- **Collapse repeated whitespace**: turns multiple spaces into one, then trims again.  

&nbsp;  
**EXAMPLE:**
&nbsp;  
1. "Beyoncé feat. JAY-Z" → "beyonce"  
2. "AC/DC" → "ac dc"  
3. " Crosby, Stills & Nash " → "crosby stills nash"  

In [9]:
def norm_artist(s: str) -> str:
    if s is None:
        return ""
    s = str(s).strip().lower()
    if not s:
        return ""
    s = _RE_FEAT.sub("", s)
    s = _strip_accents(s)
    s = _RE_NON_ALNUM.sub(" ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

#### 3.4 Normalize Titles

It normalizes a song/track title into a consistent “join key” so titles match better between Spotify and the lyrics dataset, even when one side includes extra annotations.  

- **Missing/empty guard**: if `s` is `None` or becomes empty → return `""`.  

- **Lowercase + trim**: reduces formatting differences.  

- **Remove accents**: "Señorita" → "senorita".  

- **Remove parenthetical/bracketed info**: `_RE_PARENS.sub(" ", s)`
strips things like `(Remastered 2011)`, `[Live]`, `(feat. ...)`, etc.  

- **Drop dash suffixes**: if `" - "` appears, keep only what’s before it.  

    handles Spotify-like `"Song Title - Remastered 2011"`, `"Track - Radio Edit"`, etc.  

- **Remove feature tails**: `_RE_FEAT.sub("", s)` removes `"feat..."` text if present.  

- **Standardize ampersand**: `"&"` → `" and "` so `"Simon & Garfunkel"` matches `"Simon and Garfunkel"`.  

- **Remove punctuation/symbols and keep only letters/digits/spaces**: `_RE_NON_ALNUM.sub(" ", s)`  

- **Collapse whitespace** to a single space and trim.  

&nbsp;  
**EXAMPLE:**
&nbsp;
1. "HUMBLE. (Remastered)" → "humble"  
2. "Lucky - Radio Edit" → "lucky"  
3. "Rock & Roll [Live]" → "rock and roll"  

In [10]:
def norm_title(s: str) -> str:
    if s is None:
        return ""
    s = str(s).strip().lower()
    if not s:
        return ""
    s = _strip_accents(s)
    s = _RE_PARENS.sub(" ", s)
    if " - " in s:
        s = s.split(" - ", 1)[0]
    s = _RE_FEAT.sub("", s)
    s = s.replace("&", " and ")
    s = _RE_NON_ALNUM.sub(" ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

#### 3.5 Normalize Lyrics 

It normalizes the raw lyrics text into a clean, consistent string (mainly fixing line breaks and whitespace) so downstream vectorization/features are less noisy.
 
- **Missing guard**: if `text` is `None`, return an empty string `""`.   

- **Ensure it’s a string**: `t = str(text)` (handles cases where the value isn’t already `str`).  

- **Normalize line endings**:  
    
    Converts Windows-style `\r\n` and old Mac `\r` into standard `\n`.  

- **Collapse repeated spaces/tabs**:
    
    `re.sub(r"[ \t]+", " ", t)` replaces runs of spaces and tabs with a single space.  

- **Reduce excessive blank lines**:
    
    `re.sub(r"\n{3,}", "\n\n", t)` turns 3+ consecutive newlines into just 2 (keeps paragraph breaks, removes huge gaps).  

- **Trim edges**: `strip()` removes leading/trailing whitespace/newlines.  

&nbsp;  
**EXAMPLE:**  
&nbsp;  
If the raw lyrics contain inconsistent `\r\n` and a lot of spacing, this function makes it consistent so the same lyric doesn’t look “different” just due to formatting.



In [11]:
def clean_lyrics(text: str) -> str:
    if text is None:
        return ""
    t = str(text)
    t = t.replace("\r\n", "\n").replace("\r", "\n")
    t = re.sub(r"[ \t]+", " ", t)
    t = re.sub(r"\n{3,}", "\n\n", t)
    return t.strip()

### 4. Build Spotify Lyrics Cleaned

This section builds a cleaned lyrics table from the raw Millsong dataframe, creates normalized join keys (artist_norm, title_norm), cleans the lyrics text, then deduplicates so you keep one best lyric per (artist_norm, title_norm) pair.

In [12]:
clean = df[["artist", "song", "link", "text"]].copy()
clean["artist_norm"] = clean["artist"].map(norm_artist)
clean["title_norm"] = clean["song"].map(norm_title)
clean["lyrics"] = clean["text"].map(clean_lyrics)

clean["lyrics_char_len"] = clean["lyrics"].str.len().astype("int64")

# Keep longest lyrics per (artist_norm, title_norm)
clean = (
    clean.sort_values("lyrics_char_len", ascending=False)
    .drop_duplicates(["artist_norm", "title_norm"], keep="first")
    .reset_index(drop=True)
)   

### 5. Check

In [13]:
clean.shape

(57438, 8)

In [15]:
clean[["artist", "song", "artist_norm", "title_norm", "lyrics_char_len", "lyrics"]].head(5)

,artist,song,artist_norm,title_norm,lyrics_char_len,lyrics
0,Z-Ro,Plex,z ro,plex,3925,"[Z-Ro] \nThey say he was flipping out till, lo..."
1,Bob Dylan,Bob Dylan's 115Th Dream,bob dylan,bob dylan s 115th dream,3912,I was riding on the Mayflower when I thought I...
2,Eminem,Hello,eminem,hello,3909,Hello (Hello) allow me to introduce myself (my...
3,Snoop Dogg,Doggy Dogg World,snoop dogg,doggy dogg world,3903,We'd like to welcome y'all to the fabulous Car...
4,Tom Lehrer,The Irish Ballad,tom lehrer,the irish ballad,3902,"Now I'd like to turn to the folk song, which h..."


### 6. Output

In [16]:
os.makedirs(OUTPUT_PATH, exist_ok=True)
clean.to_csv(OUTPUT_PATH + "lyrics_cleaned.csv", index=False)
print(f"Cleaned lyrics dataset saved to: {OUTPUT_PATH + 'lyrics_cleaned.csv'}")

Cleaned lyrics dataset saved to: ../data/cleaned/lyrics_cleaned.csv
